# 03. 학습곡선 · Dropout · 모델 저장
모델 생성 함수 → History → 검증곡선 → Dropout → 콜백 → 저장·복원 → 최종 평가 순서입니다.
이 노트북은 새 커널에서 독립적으로 실행합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from common.runtime import configure_runtime
from common.data import load_fashion_data

gpus = configure_runtime(seed=42)
print('GPU:', gpus)

In [ ]:
(train_x, train_y), (val_x, val_y), (test_x, test_y) = load_fashion_data()
print('훈련:', train_x.shape, train_y.shape)
print('검증:', val_x.shape, val_y.shape)
print('테스트:', test_x.shape, test_y.shape)

## 모델 생성 함수
은닉층 목록을 받아 구조를 바꿉니다. 호출할 때마다 새 계층 객체를 전달하여 가중치를 공유하지 않도록 합니다.
깊은 신경망은 여러 학습 계층을 쌓은 모델을 가리킵니다. 은닉층 하나를 추가했다고 모두 깊은 모델로 부르지는 않습니다.

In [ ]:
from pathlib import Path

def model_fn(add_layer=None):
    model = keras.Sequential([keras.Input(shape=(28, 28)), keras.layers.Flatten()])
    for layer in add_layer or []:
        model.add(layer)
    model.add(keras.layers.Dense(10, activation='softmax'))
    return model

for layers in ([], [keras.layers.Dense(100, activation='relu')],
               [keras.layers.Dense(100, activation='relu'), keras.layers.Dense(64, activation='relu')]):
    model_fn(layers).summary()

figure_dir = Path('data/figures')
model_dir = Path('data/models')
figure_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

## History와 훈련곡선
`history.epoch`는 반복 인덱스, `history.history`는 손실·정확도 기록입니다.
`verbose=0`은 출력 생략, `1`은 진행 막대, `2`는 epoch별 로그입니다.

In [ ]:
model1 = model_fn([keras.layers.Dense(64, activation='relu')])
model1.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_short = model1.fit(train_x, train_y, epochs=5, verbose=2)
print('epoch:', history_short.epoch)
print('기록 항목:', history_short.history.keys())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
epochs = np.arange(1, len(history_short.epoch) + 1)
for ax, metric in zip(axes, ['loss', 'accuracy']):
    ax.plot(epochs, history_short.history[metric])
    ax.set(xlabel='Epoch', ylabel=metric)
    ax.grid()
fig.tight_layout()
fig.savefig(figure_dir / '03_training_history.png')
plt.show()

## 검증곡선으로 과적합 살펴보기
검증 데이터는 가중치 갱신에 사용하지 않습니다. 훈련 손실이 계속 감소하는데 검증 손실이 상승하면 과대적합을 의심합니다.
고정된 손실 차이 0.1이나 차이의 부호만으로 판단하지 않습니다.
원본의 학습률 설명 0.1과 실제 코드 0.001이 달랐던 부분은 실행값 0.001로 통일했습니다.

In [ ]:
def plot_history(history, filename):
    epochs = np.arange(1, len(history.epoch) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, metric in zip(axes, ['loss', 'accuracy']):
        ax.plot(epochs, history.history[metric], label='train')
        if 'val_' + metric in history.history:
            ax.plot(epochs, history.history['val_' + metric], label='validation')
        ax.set(xlabel='Epoch', ylabel=metric)
        ax.legend()
        ax.grid()
    fig.tight_layout()
    fig.savefig(figure_dir / filename)
    plt.show()

EPOCHS = 10  # 원본 검증곡선 실험은 50회
baseline = model_fn([keras.layers.Dense(100, activation='relu')])
baseline.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
baseline_history = baseline.fit(train_x, train_y, epochs=EPOCHS,
    validation_data=(val_x, val_y), verbose=2)
plot_history(baseline_history, '03_baseline.png')

## Dropout과 콜백
Dropout(0.3)은 학습 중 활성값 일부를 확률 30%로 0으로 만듭니다. 추론 중에는 적용하지 않습니다.
학습 파라미터가 없어도 학습에 영향을 주는 규제 방법이며, 과소적합을 악화시킬 수도 있습니다.

- ModelCheckpoint: `save_best_only=True`로 검증 손실이 개선될 때 모델을 저장합니다.
- EarlyStopping: 개선이 없는 epoch가 `patience`만큼 이어지면 종료합니다.
- `restore_best_weights=True`는 최상의 가중치를 복원합니다. 하이퍼파라미터를 복원하는 옵션은 아닙니다.
- 여기서는 `val_loss`를 감시하므로 `validation_data`가 필요합니다. 모든 콜백이 검증 데이터를 요구하는 것은 아닙니다.

In [ ]:
dropout_model = model_fn([
    keras.layers.Dense(100, activation='relu'), keras.layers.Dropout(0.3)])
dropout_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
checkpoint = keras.callbacks.ModelCheckpoint(
    str(model_dir / 'best_model.keras'), monitor='val_loss', mode='min', save_best_only=True)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', mode='min', patience=2, restore_best_weights=True)
dropout_history = dropout_model.fit(train_x, train_y, epochs=50,
    validation_data=(val_x, val_y), callbacks=[checkpoint, early_stopping], verbose=2)
plot_history(dropout_history, '03_dropout.png')
best_model = keras.models.load_model(model_dir / 'best_model.keras')
print('최저 검증 손실 epoch:', np.argmin(dropout_history.history['val_loss']) + 1)
print('최적 모델 검증:', best_model.evaluate(val_x, val_y, verbose=0))

## 가중치 저장과 전체 모델 저장
가중치 파일 `.weights.h5`는 동일한 구조를 먼저 만든 뒤 불러옵니다.
전체 모델 `.keras`는 구조·가중치·저장된 옵티마이저 상태를 포함합니다.
`load_weights`는 추론에 사용할 수 있으며 재학습하려면 별도의 compile 설정도 확인해야 합니다.

In [ ]:
best_model.save_weights(model_dir / 'fashion.weights.h5')
best_model.save(model_dir / 'fashion.keras')

weights_model = model_fn([
    keras.layers.Dense(100, activation='relu'), keras.layers.Dropout(0.3)])
weights_model.load_weights(model_dir / 'fashion.weights.h5')
loaded_model = keras.models.load_model(model_dir / 'fashion.keras')

weights_proba = weights_model.predict(test_x[:32], verbose=0)
loaded_proba = loaded_model.predict(test_x[:32], verbose=0)
np.testing.assert_allclose(weights_proba, loaded_proba, rtol=1e-5, atol=1e-6)
print('두 복원 방법의 예측 일치 확인')

## 최종 테스트와 예측 해석
튜닝은 검증 데이터로 끝내고, 저장·복원한 최종 모델만 테스트합니다.
softmax 결과는 `(샘플 수, 10)`이며 `argmax(axis=1)`로 각 샘플의 예측 범주를 얻습니다.

In [ ]:
test_loss, test_accuracy = loaded_model.evaluate(test_x, test_y, verbose=0)
probabilities = loaded_model.predict(test_x, verbose=0)
predicted = np.argmax(probabilities, axis=1)
correct = int(np.sum(predicted == test_y))
incorrect = len(test_y) - correct
print('예측 모양:', probabilities.shape)
print('첫 샘플 확률:', probabilities[0])
print('첫 샘플 예측/정답:', predicted[0], test_y[0])
print(f'테스트 손실: {test_loss:.4f}, 정확도: {test_accuracy:.4f}')
print(f'정답: {correct}, 오답: {incorrect}')
print(f'정답률: {correct / len(test_y):.2%}, 오답률: {incorrect / len(test_y):.2%}')

## 복습
규제 효과는 실제 검증곡선을 보고 판단합니다. Dropout이 켜진 훈련 로그와 꺼진 평가 결과는 조건이 다릅니다.
이 노트는 파일 저장 흐름도 실습하기 위해 `data/models`와 `data/figures`에 결과를 생성합니다.
다시 실행하면 같은 이름의 모델과 그림을 갱신합니다.